In [ ]:
import os
import sys
from pathlib import Path
import importlib
import torch

In [ ]:
#@title Mount Goole Drive

root_path = "/content/drive/MyDrive/MSc/Flood-Mapping"  #@param {type:"string", multiline:true}


mount_drive = True  #@param {type:"boolean"}

clone_repo = False  #@param {type:"boolean"}

if clone_repo and mount_drive:

    from google.colab import drive
    drive.mount("/content/drive")

    root_path = os.path.join(root_path, "Flood-Mapping")

    !git clone https://github.com/TAX2310/Flood-Mapping.git $root_path

    sys.path.append(os.path.join(root_path))

    from src.config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif not clone_repo and mount_drive:
    from google.colab import drive
    drive.mount("/content/drive")

    sys.path.append(os.path.join(root_path))

    from src.config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

elif clone_repo and not mount_drive:
    root_path = "Flood-Mapping"

    !git clone https://github.com/TAX2310/Flood-Mapping.git

    sys.path.append(root_path)

    from src.config import S1_CFG
    cfg = S1_CFG()

    cfg.ROOT = Path(root_path)

else:
    from src.config import S1_CFG
    cfg = S1_CFG()

    sys.path.append(os.path.join(cfg.ROOT))

cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
requirements = cfg.ROOT / "requirements.txt"
!pip install -r {requirements}

In [ ]:
import src.train.training as training
import src.test.testing as testing
import src.util.io as io
import src.util.plotting as plot

In [ ]:
learning_rates = [1e-3, 1e-4]
batch_sizes = [32, 64]
weight_decays = [0.0, 1e-5]
dropout_rates = [0.0, 0.2]

num_workers = 8

In [ ]:
for learning_rate in learning_rates:
    for batch_size in batch_sizes:
        for weight_decay in weight_decays:
            for dropout_rate in dropout_rates:
                cfg.LR = learning_rate
                cfg.BATCH_SIZE = batch_size
                cfg.WEIGHT_DECAY = weight_decay
                cfg.DROPOUT_RATE = dropout_rate
                training.train_from_file(cfg, num_workers=num_workers)

In [ ]:
plot.view_f1_iou_bar(cfg)

In [ ]:
plot.view_training_metrics(cfg)

In [ ]:
testing.select_model_to_test(cfg)

In [ ]:
import src.inference.inference as inference

samples = ["EMSR470_AOI01_46_07_2_1.tif",
           "EMSR441_AOI05_2_3_2_2.tif",
           "EMSR570_AOI02_07_03_2_1.tif"
]

results = inference.inference(cfg, cfg.S1_MODEL, samples)
results = inference.inference(cfg, cfg.S1_MODEL)
io.save_inference_results(results, "/content/drive/MyDrive/MSc/Flood-Mapping/test_results/s1_results.pt")

In [ ]:
all_results = io.load_inference_results("/content/drive/MyDrive/MSc/Flood-Mapping/test_results/s1_results.pt")
plot_metric_distribution(all_results, metric="iou", title="Distribution of IOU scores for Sentinel-1")

In [ ]:
plot.plot_s1_results(results)